# <font color="#418FDE" size="6.5" uppercase>**Capstone Automation**</font>

>Last update: 20260608.
    
By the end of this Lecture, you will be able to:
- Design a complete Python script that automates a focused mechanical engineering analysis. 
- Integrate variables, functions, control flow, data structures, files, and modules in one coherent program. 
- Verify and communicate computational results using checks, summaries, and formatted reports. 


## **1. Project Planning**

### **1.1. Analysis Goal**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Mechanical Engineers/Module_05/Lecture_B/image_01_01.jpg?v=1780932719" width="250">



>* State a specific engineering question.
>* Link calculations to design decisions.

>* Define inputs, outputs, and assumptions early
>* Match reports to the intended user

>* Link calculations to real engineering decisions
>* Define focused, useful script results



In [ ]:
#@title Python Code - Analysis Goal

# Plan a focused mechanical engineering automation goal.
# Identify inputs, assumptions, outputs, and decisions.
# Keep the script small, clear, and testable.

# Store planning details in one dictionary.
analysis_goal = {
    "system": "simply supported aluminum beam",
    "decision": "check stress and deflection limits",
    "units": "SI units: meters, newtons, pascals",
}

# Define required input values explicitly.
inputs = {
    "length_m": 1.2,
    "width_m": 0.03,
    "height_m": 0.05,
    "center_load_N": 850.0,
    "elastic_modulus_Pa": 69e9,
    "allowable_stress_Pa": 95e6,
    "allowable_deflection_m": 0.003,
}

# Compute beam section properties safely.
def beam_properties(width_m, height_m):
    area_m2 = width_m * height_m
    inertia_m4 = width_m * height_m**3 / 12
    return area_m2, inertia_m4

# Compute maximum bending stress and deflection.
def beam_results(values):
    length_m = values["length_m"]
    load_N = values["center_load_N"]
    width_m = values["width_m"]
    height_m = values["height_m"]

    modulus_Pa = values["elastic_modulus_Pa"]
    _, inertia_m4 = beam_properties(width_m, height_m)
    moment_Nm = load_N * length_m / 4
    stress_Pa = moment_Nm * (height_m / 2) / inertia_m4

    deflection_m = load_N * length_m**3 / (48 * modulus_Pa * inertia_m4)
    stress_ok = stress_Pa <= values["allowable_stress_Pa"]
    deflection_ok = deflection_m <= values["allowable_deflection_m"]
    return stress_Pa, deflection_m, stress_ok, deflection_ok

# Check that dimensions are physically meaningful.
positive_inputs = all(value > 0 for value in inputs.values())
if not positive_inputs:
    raise ValueError("All numeric beam inputs must be positive.")

# Run the planned focused analysis.
stress_Pa, deflection_m, stress_ok, deflection_ok = beam_results(inputs)
design_ok = stress_ok and deflection_ok
status_text = "PASS" if design_ok else "REVISE"

# Print a concise report for engineering communication.
print("Analysis goal:", analysis_goal["decision"])
print("System:", analysis_goal["system"])
print("Stress: {:.1f} MPa".format(stress_Pa / 1e6))
print("Deflection: {:.2f} mm".format(deflection_m * 1000))
print("Stress check:", "OK" if stress_ok else "Too high")
print("Deflection check:", "OK" if deflection_ok else "Too high")
print("Engineering decision:", status_text)



### **1.2. Analysis Workflow**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Mechanical Engineers/Module_05/Lecture_B/image_01_02.jpg?v=1780932721" width="250">



>* Plan inputs, calculations, decisions, and outputs
>* Organize scripts to mirror engineering reasoning

>* Break analysis into testable stages
>* Validate, calculate, compare, and report results

>* Plan responses for imperfect engineering data
>* Communicate decisions through traceable workflows



In [ ]:
#@title Python Code - Analysis Workflow

# Plan each engineering automation stage.
# Validate inputs before computing results.
# Summarize decisions with traceable outputs.

import csv
import math
import tempfile
from pathlib import Path

# Define compact design cases internally.
design_cases = [
    {"case": "A", "radius_m": 0.35, "thickness_m": 0.006},
    {"case": "B", "radius_m": 0.35, "thickness_m": 0.004},
]

# Store shared engineering assumptions.
pressure_pa = 900000.0
allowable_pa = 95000000.0
required_fos = 2.0

# Create a small temporary input file.
work_folder = Path(tempfile.gettempdir())
input_file = work_folder / "vessel_cases.csv"
field_names = ["case", "radius_m", "thickness_m"]

# Write cases to demonstrate file input.
with input_file.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=field_names)
    writer.writeheader()
    writer.writerows(design_cases)

# Load cases from the prepared file.
with input_file.open("r", newline="") as handle:
    loaded_cases = list(csv.DictReader(handle))

# Validate one pressure vessel case.
def validate_case(item):
    radius = float(item["radius_m"])
    thickness = float(item["thickness_m"])
    if radius <= 0 or thickness <= 0:
        raise ValueError("Dimensions must be positive")
    if thickness >= radius:
        raise ValueError("Thickness must stay below radius")
    return radius, thickness

# Compute hoop stress and factor safety.
def analyze_case(item):
    radius, thickness = validate_case(item)
    hoop_stress = pressure_pa * radius / thickness
    factor_safety = allowable_pa / hoop_stress
    passed = factor_safety >= required_fos
    return item["case"], hoop_stress, factor_safety, passed

# Run the planned workflow stages.
results = [analyze_case(item) for item in loaded_cases]
passing_cases = sum(1 for item in results if item[3])
workflow_ok = len(results) == len(loaded_cases)

# Print concise traceable report.
print("Pressure vessel screening workflow")
print(f"Input cases loaded: {len(loaded_cases)}")
print(f"Validation and analysis completed: {workflow_ok}")

# Print one formatted line per case.
for name, stress, fos, passed in results:
    status = "PASS" if passed else "FAIL"
    stress_mpa = stress / 1_000_000.0
    print(f"Case {name}: {stress_mpa:.1f} MPa, FOS {fos:.2f}, {status}")

# Print final decision summary.
print(f"Passing designs: {passing_cases} of {len(results)}")
print("Workflow stages: input, validation, calculation, decision, report")



### **1.3. Analysis Scope**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Mechanical Engineers/Module_05/Lecture_B/image_01_03.jpg?v=1780932723" width="250">



>* Set clear boundaries for the analysis task
>* Define inputs, outputs, assumptions, and exclusions

>* Match detail to the engineering decision
>* State limits so results are trustworthy

>* Define user inputs, units, and cases
>* Plan outputs, checks, warnings, and summaries



In [ ]:
#@title Python Code - Analysis Scope

# Scope turns broad goals into bounded automation.
# This example plans one focused shaft check.
# Outputs explain assumptions, checks, and limitations.

# Define project scope as clear text items.
scope = {
    "system": "solid circular transmission shaft",
    "loads": "steady torque only",

    "inputs": "torque, diameter, strength, safety factor",
    "outputs": "shear stress, allowable stress, pass status",
}

# Store one small engineering design case.
case = {
    "torque_Nm": 420.0,
    "diameter_mm": 32.0,

    "yield_strength_MPa": 250.0,
    "safety_factor": 2.0,
}

# Define a reusable shaft stress function.
def shaft_shear_stress_mpa(torque_Nm, diameter_mm):
    torque_Nmm = torque_Nm * 1000.0
    stress = 16.0 * torque_Nmm

    stress = stress / (3.14159 * diameter_mm ** 3)
    return stress

# Verify required inputs before calculating results.
required = ["torque_Nm", "diameter_mm", "yield_strength_MPa", "safety_factor"]
missing = [key for key in required if key not in case]
if missing:
    raise ValueError("Missing required shaft inputs.")

# Check inputs stay inside intended scope.
valid_diameter = 10.0 <= case["diameter_mm"] <= 100.0
valid_factor = case["safety_factor"] >= 1.0
if not (valid_diameter and valid_factor):
    raise ValueError("Input values are outside planned scope.")

# Calculate stress and allowable design limit.
stress_MPa = shaft_shear_stress_mpa(case["torque_Nm"], case["diameter_mm"])
allowable_MPa = case["yield_strength_MPa"] / case["safety_factor"]
status = "PASS" if stress_MPa <= allowable_MPa else "REVIEW"

# Print a compact engineering scope summary.
print("Analysis scope: shaft torque screening only")
print(f"System: {scope['system']}")
print(f"Included loads: {scope['loads']}")
print(f"Stress: {stress_MPa:.1f} MPa")

# Print the decision and important limitations.
print(f"Allowable: {allowable_MPa:.1f} MPa")
print(f"Decision: {status}")
print("Excludes bending, fatigue, keyways, and vibration")
print("Use detailed analysis before final approval")



## **2. Integrated Engineering Code**

### **2.1. Functions and Modules**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Mechanical Engineers/Module_05/Lecture_B/image_02_01.jpg?v=1780932731" width="250">



>* Functions organize engineering calculations into clear tasks
>* Reusable logic makes programs readable and maintainable

>* Modules organize reusable engineering tools
>* Main scripts stay focused and consistent

>* Design functions with clear data flow
>* Test parts, then communicate engineering steps



In [ ]:
#@title Python Code - Functions and Modules

# Functions organize repeated engineering calculations.
# Modules provide reusable helper capabilities.
# This script automates shaft safety checks.

# No additional package installation is needed.
import csv
from pathlib import Path
from math import pi

# Define a small material database.
MATERIALS = {
    "AISI 1020": {"yield_mpa": 350},
    "Al 6061-T6": {"yield_mpa": 276},
}

# Store compact shaft design cases.
CASES = [
    {"name": "Case A", "diameter_mm": 28, "torque_nm": 180, "bending_nm": 90, "material": "AISI 1020"},
    {"name": "Case B", "diameter_mm": 22, "torque_nm": 160, "bending_nm": 75, "material": "Al 6061-T6"},
    {"name": "Case C", "diameter_mm": 30, "torque_nm": 220, "bending_nm": 110, "material": "AISI 1020"},
]

# Compute section properties for circular shafts.
def section_properties(diameter_mm):
    radius_m = (diameter_mm / 1000) / 2
    area_m2 = pi * radius_m**2
    polar_m4 = pi * radius_m**4 / 2
    inertia_m4 = pi * radius_m**4 / 4
    return area_m2, polar_m4, inertia_m4, radius_m

# Compute bending, torsion, and von Mises stress.
def shaft_stresses(case_data):
    _, polar_m4, inertia_m4, radius_m = section_properties(case_data["diameter_mm"])
    bending_pa = case_data["bending_nm"] * radius_m / inertia_m4
    torsion_pa = case_data["torque_nm"] * radius_m / polar_m4
    von_mises_pa = (bending_pa**2 + 3 * torsion_pa**2) ** 0.5
    return bending_pa / 1e6, torsion_pa / 1e6, von_mises_pa / 1e6

# Evaluate safety factor using material strength.
def evaluate_case(case_data):
    bend_mpa, torsion_mpa, vm_mpa = shaft_stresses(case_data)
    yield_mpa = MATERIALS[case_data["material"]]["yield_mpa"]
    safety_factor = yield_mpa / vm_mpa
    status = "PASS" if safety_factor >= 2 else "REVIEW"
    return case_data["name"], vm_mpa, safety_factor, status

# Write a compact CSV report file.
def write_report(results, path):
    with path.open("w", newline="") as file_object:
        writer = csv.writer(file_object)
        writer.writerow(["case", "von_mises_mpa", "safety_factor", "status"])
        writer.writerows(results)

# Run the capstone-style analysis workflow.
results = [evaluate_case(case_data) for case_data in CASES]
report_path = Path("shaft_safety_report.csv")
write_report(results, report_path)

# Verify the report exists before summarizing.
if not report_path.exists():
    raise FileNotFoundError("Expected report file was not created.")

# Print concise formatted engineering results.
print("Integrated shaft safety automation")
print(f"Analyzed cases: {len(results)}")
for name, vm_mpa, sf_value, status in results:
    print(f"{name}: VM={vm_mpa:.1f} MPa, SF={sf_value:.2f}, {status}")
print(f"Report file: {report_path.name}")



### **2.2. Modular Program Flow**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Mechanical Engineers/Module_05/Lecture_B/image_02_02.jpg?v=1780932733" width="250">



>* Organize scripts into clear analysis stages
>* Trace errors and improve engineering reliability

>* Combine Python parts into one workflow
>* Use modules for reusable engineering tools

>* Modular code adapts with less disruption
>* Clear flow supports reliable engineering decisions



In [ ]:
#@title Python Code - Modular Program Flow

# Demonstrate modular engineering automation flow.
# Keep each function focused and testable.
# Summarize beam deflection results professionally.

import csv
from pathlib import Path
from math import isfinite

# Define a tiny input file path.
input_path = Path("beam_cases.csv")
output_path = Path("beam_report.txt")

# Store compact built-in case data.
cases = [
    ["A36 steel", 200, 2.0, 200e9, 8.0e-6],
    ["Aluminum", 120, 1.5, 69e9, 5.5e-6],
]

# Write the input data file.
with input_path.open("w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["material", "load_N", "length_m", "E_Pa", "I_m4"])
    writer.writerows(cases)

# Read cases from a CSV file.
def read_cases(path):
    with path.open(newline="") as file:
        return list(csv.DictReader(file))

# Validate one engineering record.
def valid_case(row):
    values = [row["load_N"], row["length_m"], row["E_Pa"], row["I_m4"]]
    numbers = [float(value) for value in values]
    return all(isfinite(number) and number > 0 for number in numbers)

# Calculate cantilever tip deflection.
def tip_deflection(row):
    load = float(row["load_N"])
    length = float(row["length_m"])
    modulus = float(row["E_Pa"])
    inertia = float(row["I_m4"])
    return load * length**3 / (3 * modulus * inertia)

# Build formatted report lines.
def build_report(rows, limit_mm):
    lines = ["Beam deflection screening report"]
    for row in rows:
        if not valid_case(row):
            lines.append(f"{row['material']}: skipped invalid data")
            continue
        deflection_mm = 1000 * tip_deflection(row)
        status = "PASS" if deflection_mm <= limit_mm else "REVIEW"
        lines.append(f"{row['material']}: {deflection_mm:.3f} mm, {status}")
    return lines

# Run the modular workflow.
beam_rows = read_cases(input_path)
report_lines = build_report(beam_rows, limit_mm=2.0)
output_path.write_text("\n".join(report_lines))

# Print a concise professional summary.
print("Modular flow: read, validate, calculate, report.")
print(f"Cases processed: {len(beam_rows)}")
print(f"Report saved to: {output_path.name}")
print("\n".join(report_lines))



### **2.3. Modular Program Flow**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Mechanical Engineers/Module_05/Lecture_B/image_02_03.jpg?v=1780932735" width="250">



>* Organize analyses into clear connected stages
>* Use modular flow for readable, testable code

>* Structure decisions, loops, and repeated analyses
>* Organize data movement with modular tools

>* Check data and results at each stage
>* Build readable, extendable engineering automation tools



In [ ]:
#@title Python Code - Modular Program Flow

# This script demonstrates modular engineering automation.
# Each function owns one workflow step.
# Results are checked before reporting.

import csv
import math
from pathlib import Path

# Store one compact input file.
input_path = Path("shaft_load_cases.csv")
report_path = Path("shaft_report.txt")
rows = [

    ["case", "torque_Nm", "diameter_mm", "allowable_MPa"],
    ["startup", "120", "32", "85"],
    ["steady", "75", "28", "85"],
    ["overload", "180", "30", "85"],
]

# Write tiny data for repeatable demonstration.
with input_path.open("w", newline="") as file:
    writer = csv.writer(file)
    writer.writerows(rows)

# Read load cases from the file.
def read_cases(path):
    with path.open("r", newline="") as file:
        reader = csv.DictReader(file)
        cases = list(reader)

    if len(cases) == 0:
        raise ValueError("No load cases found.")
    return cases

# Convert text fields into engineering numbers.
def prepare_case(row):
    case = {
        "name": row["case"],
        "torque_Nm": float(row["torque_Nm"]),
        "diameter_mm": float(row["diameter_mm"]),
        "allowable_MPa": float(row["allowable_MPa"]),
    }

    if min(case["torque_Nm"], case["diameter_mm"]) <= 0:
        raise ValueError("Torque and diameter must be positive.")
    return case

# Compute torsional shear stress for solid shaft.
def calculate_stress(case):
    torque_Nmm = case["torque_Nm"] * 1000.0
    diameter = case["diameter_mm"]
    stress_MPa = 16.0 * torque_Nmm / (math.pi * diameter**3)
    return stress_MPa

# Classify each result against the allowable value.
def classify_case(case, stress_MPa):
    margin = case["allowable_MPa"] / stress_MPa
    status = "PASS" if margin >= 1.0 else "REVIEW"
    result = {"name": case["name"], "stress": stress_MPa}
    result.update({"margin": margin, "status": status})
    return result

# Run the modular workflow in order.
def analyze_cases(path):
    raw_cases = read_cases(path)
    prepared = [prepare_case(row) for row in raw_cases]
    results = []

    for case in prepared:
        stress = calculate_stress(case)
        results.append(classify_case(case, stress))
    return results

# Build a concise report for communication.
def write_report(path, results):
    lines = ["Shaft torsion automation summary"]
    for item in results:
        line = f"{item['name']}: {item['stress']:.1f} MPa"
        lines.append(f"{line}, margin {item['margin']:.2f}, {item['status']}")

    path.write_text("\n".join(lines))
    return lines

# Execute analysis and report key checkpoints.
results = analyze_cases(input_path)
report_lines = write_report(report_path, results)
failed = sum(item["status"] == "REVIEW" for item in results)

# Print a short verification summary.
print("Modular shaft analysis completed.")
print(f"Cases analyzed: {len(results)}.")
print(f"Cases requiring review: {failed}.")
print(report_lines[0])

# Print compact per-case findings.
for line in report_lines[1:]:
    print(line)

print(f"Report file written: {report_path.name}.")



## **3. Result Review**

### **3.1. Validation Checks**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Mechanical Engineers/Module_05/Lecture_B/image_03_01.jpg?v=1780932729" width="250">



>* Check automated results for engineering sense
>* Catch errors before trusting calculated outputs

>* Check ranges, consistency, units, and scale
>* Compare results with trusted estimates

>* Report checks, warnings, and assumptions clearly
>* Support confidence, collaboration, and responsible decisions



In [ ]:
#@title Python Code - Validation Checks

# Review automated beam deflection results.
# Validation checks protect engineering decisions.
# Short reports communicate calculation confidence.

# Define beam input data in SI units.
beam = {"length_m": 2.0, "load_n": 850.0}
beam["modulus_pa"] = 200e9
beam["inertia_m4"] = 8.0e-6

# Compute cantilever tip deflection magnitude.
def cantilever_deflection(load_n, length_m, modulus_pa, inertia_m4):
    numerator = load_n * length_m ** 3
    denominator = 3 * modulus_pa * inertia_m4

    return numerator / denominator

# Compute stress using a simple estimate.
def bending_stress(load_n, length_m, section_modulus_m3):
    moment_nm = load_n * length_m
    return moment_nm / section_modulus_m3

# Store derived values for review.
deflection_m = cantilever_deflection(
    beam["load_n"], beam["length_m"],
    beam["modulus_pa"], beam["inertia_m4"]
)

# Estimate stress using section modulus.
section_modulus_m3 = 1.6e-4
stress_pa = bending_stress(
    beam["load_n"], beam["length_m"], section_modulus_m3
)

# Define engineering validation limits.
allowable_stress_pa = 140e6
small_deflection_limit_m = beam["length_m"] / 100
expected_deflection_m = 0.00142

# Collect validation checks as messages.
checks = []
checks.append(("Positive dimensions", beam["length_m"] > 0))
checks.append(("Small deflection", deflection_m < small_deflection_limit_m))
checks.append(("Stress allowable", stress_pa < allowable_stress_pa))

# Check result against a benchmark estimate.
relative_error = abs(deflection_m - expected_deflection_m) / expected_deflection_m
checks.append(("Benchmark agreement", relative_error < 0.10))

# Summarize pass and fail counts.
passed = sum(1 for name, ok in checks if ok)
failed = len(checks) - passed
status = "READY FOR REVIEW" if failed == 0 else "INVESTIGATE"

# Print a compact engineering report.
print("Beam validation report")
print(f"Deflection: {deflection_m*1000:.3f} mm")
print(f"Stress: {stress_pa/1e6:.2f} MPa")
print(f"Checks passed: {passed}/{len(checks)}")

# Print each validation result briefly.
for name, ok in checks:
    label = "PASS" if ok else "FAIL"
    print(f"{label}: {name}")

# Print final result status.
print(f"Final status: {status}")



### **3.2. Formatted Reports**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Mechanical Engineers/Module_05/Lecture_B/image_03_02.jpg?v=1780932725" width="250">



>* Turn calculations into clear engineering decisions
>* Include assumptions, results, criteria, and status

>* Show units, labels, and suitable precision
>* Turn results into clear engineering decisions

>* Compare repeated cases and highlight key findings
>* Write reports carefully to support decisions



In [ ]:
#@title Python Code - Formatted Reports

# Build a concise engineering report.
# Check results before reporting them.
# Format values with useful units.

# Store shaft inputs in one dictionary.
shaft = {
    "name": "Mixer drive shaft",
    "diameter_in": 1.25,

    "moment_lbf_in": 1800.0,
    "yield_ksi": 36.0,
}

# Define stress calculation for bending.
def bending_stress_ksi(moment_lbf_in, diameter_in):
    numerator = 32.0 * moment_lbf_in

    denominator = 3.14159 * diameter_in**3
    stress_psi = numerator / denominator

    return stress_psi / 1000.0

# Compute the main engineering result.
stress_ksi = bending_stress_ksi(
    shaft["moment_lbf_in"], shaft["diameter_in"]
)

# Compute factor of safety.
factor_safety = shaft["yield_ksi"] / stress_ksi

# Check design status against requirement.
required_fos = 2.0
status = "PASS" if factor_safety >= required_fos else "FAIL"

# Build formatted report lines.
report_lines = [
    "SHAFT BENDING REVIEW",
    f"Component: {shaft['name']}",

    f"Diameter: {shaft['diameter_in']:.2f} in",
    f"Bending moment: {shaft['moment_lbf_in']:.0f} lbf-in",
    f"Allowable yield strength: {shaft['yield_ksi']:.1f} ksi",

    f"Calculated stress: {stress_ksi:.2f} ksi",
    f"Factor of safety: {factor_safety:.2f}",
    f"Required factor of safety: {required_fos:.2f}",

    f"Design status: {status}",
    "Note: Units and precision are stated explicitly.",
]

# Print the concise formatted report.
print("\n".join(report_lines))



### **3.3. Formatted Reports**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Mechanical Engineers/Module_05/Lecture_B/image_03_03.jpg?v=1780932727" width="250">



>* Turn raw results into understandable engineering reports
>* Summarize inputs, assumptions, results, and decisions

>* Use clear labels, units, and context
>* Round results to match decision needs

>* Document assumptions, checks, and conclusions clearly
>* Support comparison, review, and confident decisions



In [ ]:
#@title Python Code - Formatted Reports

# This script demonstrates formatted engineering reports.
# It summarizes a compact shaft review.
# Each printed line has useful context.

import math
from datetime import datetime

# Define required shaft design inputs.
shaft_name = "Conveyor drive shaft"
power_kw = 12.0
speed_rpm = 900.0

# Define geometry and material properties.
diameter_mm = 35.0
allowable_shear_mpa = 85.0
minimum_safety_factor = 2.0

# Convert inputs into consistent base units.
power_w = power_kw * 1000.0
speed_rad_s = speed_rpm * 2.0 * math.pi / 60.0
radius_mm = diameter_mm / 2.0

# Compute torque from transmitted power.
torque_nm = power_w / speed_rad_s
torque_nmm = torque_nm * 1000.0
polar_moment_mm4 = math.pi * diameter_mm**4 / 32.0

# Compute shear stress and safety factor.
shear_stress_mpa = torque_nmm * radius_mm / polar_moment_mm4
safety_factor = allowable_shear_mpa / shear_stress_mpa
status = "PASS" if safety_factor >= minimum_safety_factor else "FAIL"

# Store key results in a dictionary.
results = {
    "Torque": f"{torque_nm:,.1f} N·m",
    "Shear stress": f"{shear_stress_mpa:,.1f} MPa",
    "Safety factor": f"{safety_factor:,.2f}",
    "Review status": status,
}

# Print a compact formatted engineering report.
print("SHAFT DESIGN REVIEW REPORT")
print(f"Generated: {datetime.now():%Y-%m-%d %H:%M}")
print(f"Component: {shaft_name}")
print(f"Input power: {power_kw:.1f} kW at {speed_rpm:.0f} rpm")
print(f"Diameter: {diameter_mm:.1f} mm")
print(f"Allowable shear stress: {allowable_shear_mpa:.1f} MPa")
print("-" * 38)
for label, value in results.items():
    print(f"{label:<16}: {value}")
print("-" * 38)
print(f"Required safety factor: {minimum_safety_factor:.2f}")
print(f"Conclusion: design {status} based on torsional shear check.")



# <font color="#418FDE" size="6.5" uppercase>**Capstone Automation**</font>


In this lecture, you learned to:
- Design a complete Python script that automates a focused mechanical engineering analysis. 
- Integrate variables, functions, control flow, data structures, files, and modules in one coherent program. 
- Verify and communicate computational results using checks, summaries, and formatted reports. 

<font color='yellow'>Congratulations on completing this course!</font>